Before running this notbook, please run codonaligner.ipynb and filting_introns.ipynb

In [1]:
import cogent3
from cogent3 import get_app
from cogent3.maths.matrix_exponential_integration import expected_number_subs
import matplotlib.pyplot as plt
import paths
import libs
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm
import os

import trinuc_models as trinucs # this module must be in the same directory as this notebook

## CDS

In [3]:
#Setting up the rules for model fitting of cds regions
sm_noncds=trinucs.GNC_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_cds = [{"par_name": n, "is_independent": True} for n in paramnames]
GNC_subsmodel = get_app("model", "GNC_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_cds)

#Setting up the app for reading cds sequences
loader_cds = get_app("load_aligned", moltype="dna")   
omit_degs_cds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

#create a concatenated alignment with all coding positions
cds_process = loader_cds + omit_degs_cds

In [4]:

folder_in = paths.DATA_HUMCHIMPORANG115 + 'cds/chrm10/codon_aligned'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_cds = [r for r in cds_process.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

cds_alns = concat(nonconcat_cds)
print("alignment length:", len(cds_alns))
cds_alns.source = "cds_alignments"

result_cds = GNC_subsmodel(cds_alns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_cds.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_cds))

result_cds.lf

   0%|          |00:00<?

alignment length: 1156035


/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/recalculation/definition.py:690: UserWarning: using slow exponentiator because 'eigen failed precision test'
  return func(*args)


   0%|          |00:00<?

GNC_CpG_ss
log-likelihood = -1708062.3568
number of free parameters = 102
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.01               5.58    1.40    5.02    1.03
Chimpanzee    root        0.02               4.21    1.10    1.84    1.30
Orangutan     root        0.04               4.50    1.17    4.00    0.57
-------------------------------------------------------------------------

continued: 
=====================================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C    omega
---------------------------------------------------------------------
1.67    2.21    7.23    6.21    2.02    1.77    0.86    4.49     0.30
1.15    1.40    2.47    2.11    1.28    1.53    0.84    1.72     0.46
1.20    1.51    4.15    3.63    1.38    1.12    0.62    4.06     0.25
---------------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.03    0.02    0.03    0.02    0.02    0.02    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.02    0.02    0.01    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.02    0.01    0.02    0.01    0.01    0.01    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.04    0.01    0.03    0.03    0.04    0.02    0.02    0.03    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAC     TAT
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.01    0.01    0.03    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TCA     TCC     TCG     TCT     TGC     TGG     TGT     TTA     TTC     TTG
----------------------------------------------------------------------------
0.01    0.02    0.00    0.02    0.01    0.01    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
====
 TTT
----
0.02
----

# Non cds

In [2]:
#Setting up the rules for model fitting of noncds regions
sm_noncds=trinucs.GT_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_noncds = [{"par_name": n, "is_independent": True, "upper": 100} for n in paramnames if n!="omega"] + [{"par_name": "omega", "value": 1.0, "is_constant": True}]
GT_subsmodel = get_app("model", "GT_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_noncds)

#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Orangutan, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs_noncds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

noncds_app = loader + rename_noncds + omit_degs_noncds

## Intergenic Ancestral Repeats (IGAR)

In [6]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_IGAR = concat(nonconcat_IGAR)
print("alignment length:", len(alns_IGAR))
alns_IGAR.source = "IGAR_alignments"

result_IGAR = GT_subsmodel(alns_IGAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_IGAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_IGAR))

result_IGAR.lf

   0%|          |00:00<?

alignment length: 69258


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -266018.2763
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        1.73             100.00    0.14    0.20    0.00
Chimpanzee    root        1.78             100.00    0.16    0.28    0.00
Orangutan     root        3.21              21.51    0.57    0.53    0.61
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.33    0.28    0.09    0.06    0.10    0.00    1.18    1.09
0.39    0.36    0.10    0.06    0.23    0.00    1.29    1.13
0.61    0.58    0.66    0.42    0.61    0.59    1.22    1.00
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.03    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.01    0.00    0.02    0.00    0.00    0.00    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.03    0.01    0.00    0.01    0.01    0.01    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.00    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.02    0.05    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.03    0.01    0.37
----------------------------

## Introns

In [3]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'introns/chrm10/noUTRs/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_introns = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_introns = concat(nonconcat_introns)
print("alignment length:", len(alns_introns))
alns_introns.source = "introns_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_introns = GT_subsmodel(alns_introns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_introns.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_introns))

result_introns.lf

   0%|          |00:00<?

alignment length: 83646


/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/recalculation/definition.py:690: UserWarning: using slow exponentiator because 'eigen failed precision test'
  return func(*args)


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -314560.8225
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        2.05             100.00    0.00    0.05    0.00
Orangutan     root        3.14              17.89    0.36    0.44    0.46
Chimpanzee    root        2.06             100.00    0.00    0.05    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.02    0.09    0.00    0.02    0.00    1.38    1.08
0.35    0.35    0.67    0.19    0.50    0.50    1.51    1.09
0.00    0.03    0.10    0.00    0.00    0.00    1.43    1.14
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.00    0.01    0.01    0.01    0.00    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.00    0.02    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.00    0.03    0.01    0.00    0.01    0.00    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.00    0.00    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.01    0.03    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.01    0.00    0.51
----------------------------

## 5' UTR

In [5]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'introns/chrm10/5UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_5UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_5UTR = concat(nonconcat_5UTR)
print("alignment length:", len(alns_5UTR))
alns_5UTR.source = "5UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_5UTR = GT_subsmodel(alns_5UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_5UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_5UTR))

result_5UTR.lf

   0%|          |00:00<?

alignment length: 63870


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -207231.1662
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.48              13.17    0.99    0.95    1.33
Orangutan     root        1.18               7.97    0.93    1.01    1.15
Chimpanzee    root        0.65               9.25    1.13    1.11    1.52
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
1.12    0.78    0.78    0.81    0.66    0.96    1.33    0.89
1.09    1.19    1.13    1.03    1.18    1.03    1.11    1.05
1.31    1.16    1.08    1.06    0.94    1.10    1.40    1.22
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.04    0.02    0.02    0.02    0.02    0.01    0.01    0.02    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.01    0.02    0.01    0.01    0.03    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.02    0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.02    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.02    0.02    0.01    0.02    0.02    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.02    0.01    0.04
----------------------------

## 3' UTR

In [6]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'introns/chrm10/3UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_3UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_3UTR = concat(nonconcat_3UTR)
print("alignment length:", len(alns_3UTR))
alns_3UTR.source = "3UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_3UTR = GT_subsmodel(alns_3UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_3UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_3UTR))

result_3UTR.lf

   0%|          |00:00<?

alignment length: 28743


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -106578.5836
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        1.24             100.00    0.21    0.30    0.00
Orangutan     root        2.50               9.32    0.54    0.74    0.55
Chimpanzee    root        1.33             100.00    0.34    0.49    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.30    0.29    0.08    0.01    0.08    0.00    1.41    0.92
0.57    0.78    0.75    0.46    0.75    0.61    1.40    0.99
0.40    0.45    0.09    0.01    0.25    0.00    1.72    1.10
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.01    0.02    0.01    0.01    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.04    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.04    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.00    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.03    0.01    0.01    0.01    0.04    0.00    0.00    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.03    0.03    0.01    0.27
----------------------------

## Introns Ancestral Repeats (IntronAR)

In [7]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'intronsAR/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_intronsAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_intronsAR = concat(nonconcat_intronsAR)
print("alignment length:", len(alns_intronsAR))
alns_intronsAR.source = "intronsAR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_intronsAR = GT_subsmodel(alns_intronsAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_intronsAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_intronsAR))

result_intronsAR.lf

   0%|          |00:00<?

alignment length: 73164


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -271342.1809
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        2.03             100.00    0.00    0.03    0.00
Orangutan     root        2.81              17.49    0.31    0.33    0.29
Chimpanzee    root        2.01             100.00    0.01    0.05    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.00    0.07    0.00    0.02    0.00    1.11    1.00
0.28    0.29    0.48    0.13    0.38    0.35    1.25    1.02
0.00    0.03    0.08    0.00    0.02    0.00    1.24    1.04
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.00    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.02    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.00    0.02    0.01    0.00    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.00    0.00    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.02    0.03    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.01    0.00    0.48
----------------------------

## Intergenic distal (5kb distal from transcript)

In [8]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'distalIG/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_distalIG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_distalIG = concat(nonconcat_distalIG)
print("alignment length:", len(alns_distalIG))
alns_distalIG.source = "distalIG_alignments"

result_distalIG = GT_subsmodel(alns_distalIG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_distalIG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_distalIG))

result_distalIG.lf

   0%|          |00:00<?

alignment length: 50817


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -183874.4930
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        1.03              34.88    1.06    1.07    1.31
Orangutan     root        2.39              22.22    1.07    1.00    1.36
Chimpanzee    root        1.11              24.79    1.08    0.96    1.34
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.90    0.91    0.63    0.69    0.91    0.92    1.08    1.05
1.07    1.05    0.97    0.93    1.04    1.10    1.29    1.05
0.76    0.93    0.74    0.83    0.88    0.93    1.16    1.01
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.05    0.01    0.02    0.03    0.01    0.01    0.01    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.01    0.03    0.02    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.02    0.03    0.01    0.02    0.01    0.01    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.02    0.01    0.01    0.03    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.02    0.01    0.05
----------------------------

## Intergenic proximal 5' (5kb proximal from transcript start)

In [9]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'proximal5IG/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal5IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal5IG = concat(nonconcat_proximal5IG)
print("alignment length:", len(alns_proximal5IG))
alns_proximal5IG.source = "proximal5IG_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal5IG = GT_subsmodel(alns_proximal5IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_proximal5IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal5IG))

result_proximal5IG.lf

   0%|          |00:00<?

alignment length: 19677


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -73467.3648
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Human         root        2.00             100.00    23.61    21.46    30.35
Orangutan     root        2.25              32.94     8.99     9.43    11.26
Chimpanzee    root        2.11              81.74    13.49    14.83    18.53
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    1.88    0.67    1.63    0.00    0.68    0.00    1.77
1.25    2.28    1.81    2.34    1.82    0.99    0.69    2.31
0.00    2.73    0.83    1.74    2.50    1.96    0.00    2.02
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.52    0.01    0.02    0.02    0.01    0.01    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.00    0.00    0.00    0.01    0.00    0.00    0.00    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.01    0.00    0.01    0.00    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.00    0.01    0.00    0.00    0.00    0.01    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.01    0.01    0.00    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.01    0.01
----------------------------

## Intergenic proximal 3' (5kb proximal from transcript end)

In [10]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'proximal3IG/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal3IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal3IG = concat(nonconcat_proximal3IG)
alns_proximal3IG.source = "proximal3IG_alignments"
print("alignment length:", len(alns_proximal3IG))

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal3IG = GT_subsmodel(alns_proximal3IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_proximal3IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal3IG))

result_proximal3IG.lf

   0%|          |00:00<?

alignment length: 24741


   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -92008.0128
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Human         root        2.28             100.00    26.19    26.69    29.16
Orangutan     root        2.59              38.27     6.06     6.17     7.76
Chimpanzee    root        2.23             100.00    37.50    39.05    42.74
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.48    0.04    1.72    0.17    0.00    0.00    0.49
1.60    1.21    0.09    1.20    0.82    0.80    0.97    1.60
0.00    0.07    0.02    2.47    0.00    0.00    0.00    0.26
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.55    0.01    0.02    0.02    0.00    0.01    0.01    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.02    0.00    0.01    0.01    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.01    0.01    0.01    0.01    0.01    0.01    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.01    0.00    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.00    0.01    0.00    0.00    0.00    0.01    0.01    0.02    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.01    0.00    0.00    0.01    0.01    0.01    0.00    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.01    0.01
----------------------------